# Notebook 04: Sentence Embeddings & Contrastive Training (Bi-Encoders)

Welcome to the fourth notebook in the **Embeddings from Scratch** series!

In Notebook 03, we uncovered a critical problem: **raw pre-trained BERT embeddings are anisotropic and poorly calibrated for semantic distance out of the box.**

In this notebook, we solve this by understanding **how embedding models are actually trained for retrieval and RAG**:
* The **Bi-Encoder Architecture (Sentence-BERT)**
* **Contrastive Learning & Metric Learning**
* **Triplet Loss** vs. **Multiple Negatives Ranking Loss (MNRL / InfoNCE)**
* The vital role of **Hard Negatives**
* Building a full Bi-Encoder training step from scratch in PyTorch.


---
## 1. Setup & Environment
Run this cell on your Google server / Colab instance (GPU recommended).


In [1]:
# Run this on your Google server / Colab instance:
# !pip install -q transformers torch numpy sentence-transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


---
## 2. Cross-Encoder vs. Bi-Encoder: The Scalability Bottleneck

Suppose you have a database of **1,000,000 document chunks** in your RAG system.

### Option A: Cross-Encoder
Pass both the query and document together into the Transformer:
$$\text{Score} = \text{Transformer}(\text{Query} + \text{Document})$$
* **Pros**: Incredible accuracy because every token of the query attends directly to every token of the document.
* **Cons**: To answer **one user query**, you must run the Transformer **1,000,000 times**! That would take minutes or hours per search.

### Option B: Bi-Encoder (Dense Retrieval)
Encode the query and documents **independently** into single fixed-size vectors:
$$\mathbf{u} = \text{Encoder}(\text{Query}), \quad \mathbf{v} = \text{Encoder}(\text{Document})$$
$$\text{Score} = \cos(\mathbf{u}, \mathbf{v}) = \mathbf{u} \cdot \mathbf{v}$$
* **Pros**: All 1,000,000 documents are embedded **once offline** and indexed in a vector database. At query time, you only embed **one query**, followed by a millisecond dot-product search!
* **Cons**: Slightly lower accuracy than Cross-Encoder because the query and document cannot cross-attend to each other.

```
       Cross-Encoder                       Bi-Encoder (SBERT)
       -------------                       ------------------
   [Query] + [Document]                    [Query]       [Document]
            │                                 │               │
     [Transformer]                     [Transformer]   [Transformer]
            │                          (shared weights, or Siamese)
      [Relevance Score]                       │               │
   (Unusable for 1M docs)               [Vector u]       [Vector v]
                                              └───────┬───────┘
                                                Dot Product
                                            (Sub-millisecond!)
```


---
## 3. Contrastive Learning: The Intuition

In classification, we have discrete classes (Dog, Cat, Bird).
In representation learning for RAG, we don't have static classes; we have **relationships**:
* A **Query** should be close to its **Relevant Answer (Positive)**.
* A **Query** should be far from **Irrelevant Documents (Negatives)**.

```
                         Vector Space
       Before Training                  After Contrastive Training
       
     [Doc Neg 1]                           [Doc Neg 1]
         ▲
         │                                               [Doc Neg 2]
      [Query] ──► [Doc Neg 2]               [Query]
         │                                     ▲ (Pulled close)
         ▼                                     │
    [Doc Positive]                         [Doc Positive]
(Random / Anisotropic)             (Semantics = Geometric Proximity)
```


---
## 4. Loss Functions for Embedding Training

### 4.1 Triplet Loss
A dataset of triplets: $(\text{Anchor } a, \text{Positive } p, \text{Negative } n)$.
We want the distance $D(a, p)$ to be smaller than $D(a, n)$ by at least a safety margin $\epsilon$:

$$\mathcal{L}_{\text{triplet}} = \max(0, D(a, p) - D(a, n) + \epsilon)$$

* If $D(a, n) > D(a, p) + \epsilon$, loss is $0$ (already well-separated).
* Otherwise, the gradient pushes $p$ closer to $a$ and repels $n$ further from $a$.

### 4.2 Multiple Negatives Ranking Loss (MNRL / InfoNCE)
Modern state-of-the-art embedding models (OpenAI, BGE, E5, Cohere) use **MNRL**.

Instead of manually constructing negative triplets for every pair, we feed a batch of $B$ positive pairs $(q_i, p_i)$.
For query $q_i$:
* $p_i$ is the **positive passage**.
* All other $B - 1$ passages in the same batch ($p_j$ where $j \neq i$) serve as **in-batch negatives**!

$$\mathcal{L}_i = -\log \frac{\exp(\text{sim}(q_i, p_i) / \tau)}{\sum_{j=1}^B \exp(\text{sim}(q_i, p_j) / \tau)}$$
where $\tau$ is a temperature parameter (typically $0.05$ or $0.07$).

Notice that this is identical to **Cross-Entropy Loss**, where the target class is the diagonal index $i$!


---
## 5. Implementing a Bi-Encoder and MNRL from Scratch

Let's implement a complete Bi-Encoder model with Mean Pooling and the Multiple Negatives Ranking Loss function in PyTorch.


In [2]:
class BiEncoder(nn.Module):
    def __init__(self, model_name="bert-base-uncased"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        
    def mean_pooling(self, token_embeddings, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling across tokens
        embeddings = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        # Normalize to unit length (L2 norm = 1)
        return F.normalize(embeddings, p=2, dim=1)

class MultipleNegativesRankingLoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature
        self.cross_entropy = nn.CrossEntropyLoss()
        
    def forward(self, query_embeddings, doc_embeddings):
        """
        query_embeddings: [batch_size, hidden_dim] (unit normalized)
        doc_embeddings:   [batch_size, hidden_dim] (unit normalized)
        """
        # Compute cosine similarity matrix: [batch_size, batch_size]
        similarity_matrix = torch.matmul(query_embeddings, doc_embeddings.T) / self.temperature
        
        # Ground truth labels: Diagonal elements (query_i matches doc_i)
        batch_size = query_embeddings.size(0)
        labels = torch.arange(batch_size, device=query_embeddings.device)
        
        return self.cross_entropy(similarity_matrix, labels)

print("BiEncoder and MultipleNegativesRankingLoss defined!")


BiEncoder and MultipleNegativesRankingLoss defined!


---
## 6. Training on a Toy Retrieval Dataset

Let's prepare a toy dataset with queries and their matching positive documents:


In [3]:
# Sample query-positive pairs
training_data = [
    ("How does photosyntheis work in plants?", 
     "Plants absorb sunlight and water to produce glucose and oxygen via chlorophyll."),
    
    ("What causes thunder and lightning?", 
     "Lightning is a rapid electrostatic discharge between clouds, producing sound waves known as thunder."),
     
    ("Who invented the telephone?", 
     "Alexander Graham Bell was awarded the first US patent for the electric telephone in 1876."),
     
    ("How to make homemade Italian pasta?", 
     "Traditional Italian pasta dough consists of semolina flour, fresh eggs, and a pinch of salt.")
]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = BiEncoder("bert-base-uncased").to(device)
criterion = MultipleNegativesRankingLoss(temperature=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print(f"Batch size: {len(training_data)} queries and positive passages.")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batch size: 4 queries and positive passages.


### 6.1 The Training Step


In [4]:
queries = [item[0] for item in training_data]
docs    = [item[1] for item in training_data]

# Tokenize queries and docs separately
q_inputs = tokenizer(queries, padding=True, truncation=True, return_tensors="pt").to(device)
d_inputs = tokenizer(docs,    padding=True, truncation=True, return_tensors="pt").to(device)

model.train()
EPOCHS = 15

print("Starting Bi-Encoder Contrastive Training...")
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    
    # Encode queries and docs independently
    q_vecs = model(q_inputs['input_ids'], q_inputs['attention_mask'])
    d_vecs = model(d_inputs['input_ids'], d_inputs['attention_mask'])
    
    loss = criterion(q_vecs, d_vecs)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 3 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], MNRL Loss: {loss.item():.4f}")


Starting Bi-Encoder Contrastive Training...
Epoch [3/15], MNRL Loss: 0.0060
Epoch [6/15], MNRL Loss: 0.0002
Epoch [9/15], MNRL Loss: 0.0000
Epoch [12/15], MNRL Loss: 0.0000
Epoch [15/15], MNRL Loss: 0.0000


---
## 7. Evaluating the Learned Metric: The Hard Negatives Problem

Now, let's see how our trained embeddings behave.
We'll test a query with:
1. Its **True Positive**
2. A **Hard Negative** (contains the exact same keywords like `"telephone"`, but does NOT answer the query!)
3. An **Easy Negative** (completely unrelated topic)


In [5]:
test_query = "Who invented the telephone?"

candidates = [
    "Alexander Graham Bell was awarded the first US patent for the electric telephone in 1876.", # Positive
    "The telephone company charges exorbitant fees for international cellular roaming data.",   # Hard Negative
    "Chocolate chip cookies are baked at 350 degrees Fahrenheit for 12 minutes."                # Easy Negative
]

model.eval()
with torch.no_grad():
    q_enc = model(tokenizer(test_query, return_tensors="pt")['input_ids'].to(device),
                  tokenizer(test_query, return_tensors="pt")['attention_mask'].to(device))
    
    c_inputs = tokenizer(candidates, padding=True, return_tensors="pt").to(device)
    c_enc = model(c_inputs['input_ids'], c_inputs['attention_mask'])
    
    # Cosine similarities (dot products because vectors are unit normalized)
    scores = torch.matmul(q_enc, c_enc.T).squeeze(0).cpu().numpy()

for text, score in zip(candidates, scores):
    print(f"Score: {score:+.4f} | Candidate: {text}")


Score: +0.8439 | Candidate: Alexander Graham Bell was awarded the first US patent for the electric telephone in 1876.
Score: +0.3996 | Candidate: The telephone company charges exorbitant fees for international cellular roaming data.
Score: +0.2410 | Candidate: Chocolate chip cookies are baked at 350 degrees Fahrenheit for 12 minutes.


### Why Hard Negatives Matter:
* Notice how a naive model might give the **Hard Negative** high similarity simply because it repeats the word `"telephone"`.
* In production retrieval models (like **BGE** or **E5**), **Hard Negative Mining** (using BM25 or Cross-Encoders to find confusing documents during training) is the secret to building world-class embeddings!

---
## Summary of Notebook 04
1. **Bi-Encoders** allow precomputing document embeddings offline, enabling sub-millisecond search over millions of documents.
2. **Contrastive Learning** pulls matching query-document pairs together and pushes negatives apart.
3. **Multiple Negatives Ranking Loss (MNRL)** uses other batch elements as negative samples, drastically improving sample efficiency.
4. **Hard Negatives** prevent the model from relying on superficial keyword overlap, teaching true semantic reasoning.

In **Notebook 05**, we will explore **Modern Embedding Architectures for RAG**: **Dense vs. Sparse (SPLADE)**, **Late Interaction (ColBERT)**, and **Matryoshka Representation Learning (MRL)**!
